<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/mates/notebooks/c3_l6.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C3-L6 · Sharpe y Sortino | Retorno por unidad de susto sobre 50 retornos diarios.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red ), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c3_l6.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/mates/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
import numpy as np
r = df["ret"]
sharpe = r.mean() / r.std(ddof=1) * (252 ** 0.5)
down = r[r < 0]
sortino = r.mean() / down.std(ddof=1) * (252 ** 0.5)
equity = np.exp(r.cumsum())
ret_total = equity.iloc[-1] - 1
mdd = ((equity / equity.cummax()) - 1).min()
print(f"Sharpe anualizado: {sharpe:.2f}")
print(f"Sortino anualizado: {sortino:.2f}")
print(f"Retorno total: {ret_total:.2%} | MaxDD: {mdd:.2%}")
print(f"Días en rojo: {(r < 0).sum()} de {len(r)}")

## El precio del retorno | El equity sube 8,32% pero pasa por un pozo de −11,81%.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(equity.values, color="#5eead4", label="equity")
ax.plot(equity.cummax().values, color="#f59e0b", linestyle="--", label="pico previo")
ax.set_ylabel("equity (base 1)")
ax.set_title("Equity con drawdown: Sharpe 1,31, MaxDD −11,81%")
ax.legend()
plt.show()

In [ ]:
print("Sharpe castiga toda la volatilidad; Sortino solo la de los días rojos.")
print(f"Por eso aquí Sortino ({sortino:.2f}) duplica al Sharpe ({sharpe:.2f}).")

In [ ]:
# Chequeos automáticos
assert len(df) == 50, "se esperan 50 días"
assert 0.9 < sharpe < 1.4, "Sharpe determinista ≈ 1,31"
assert sortino > sharpe, "el Sortino debe superar al Sharpe en esta serie"
assert mdd < 0, "toda equity real tiene un drawdown"
print("Chequeos OK: Sharpe 1,31 / Sortino 2,60 / MaxDD −11,81%")